In [2]:
## fix  ###
import os
import time
import shap
import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedKFold, learning_curve, KFold
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import QuantileTransformer, StandardScaler
from sklearn.utils import resample
from rdkit import Chem
from rdkit.Chem import GraphDescriptors, MolSurf, Descriptors, Lipinski, rdMolDescriptors, EState
from rdkit.Chem.EState import EState_VSA
from rdkit.Chem.rdchem import Mol
from sklearn.linear_model import ElasticNet
from sklearn.feature_selection import mutual_info_regression
from scipy.cluster import hierarchy
from sklearn.tree import DecisionTreeRegressor
import joblib
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import pearsonr, gaussian_kde
import seaborn as sns

# Set global font settings for matplotlib
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.unicode_minus'] = False

def molecular_descriptors(mol: Mol) -> dict:
    descriptors = {}

    # Original descriptors
    descriptors['Kappa1'] = GraphDescriptors.Kappa1(mol)
    descriptors['Kappa2'] = GraphDescriptors.Kappa2(mol)
    descriptors['Kappa3'] = GraphDescriptors.Kappa3(mol)
    descriptors['BertzCT'] = GraphDescriptors.BertzCT(mol)

    # PEOE_VSA descriptors
    for i in range(1, 15):
        descriptors[f'PEOE_VSA{i}'] = getattr(MolSurf, f'PEOE_VSA{i}')(mol)

    # Other descriptors
    descriptors['heavy_atom_count'] = Descriptors.HeavyAtomCount(mol)
    descriptors['rotatable_bonds'] = Descriptors.NumRotatableBonds(mol)
    descriptors['h_acceptors'] = Descriptors.NumHAcceptors(mol)
    descriptors['heteroatom_count'] = Descriptors.NumHeteroatoms(mol)
    descriptors['tpsa'] = Descriptors.TPSA(mol)
    descriptors['FpDensity'] = Descriptors.FpDensityMorgan1(mol)
    descriptors['FractionCSP3'] = Lipinski.FractionCSP3(mol)

    # Chi descriptors
    for i in range(5):
        descriptors[f'Chi{i}n'] = getattr(rdMolDescriptors, f'CalcChi{i}n')(mol)
        descriptors[f'Chi{i}v'] = getattr(rdMolDescriptors, f'CalcChi{i}v')(mol)

    # Crippen descriptors
    logp, mr = rdMolDescriptors.CalcCrippenDescriptors(mol)
    descriptors['CrippenLogP'] = logp
    descriptors['CrippenMR'] = mr

    descriptors['ExactMolWt'] = rdMolDescriptors.CalcExactMolWt(mol)
    descriptors['HallKierAlpha'] = rdMolDescriptors.CalcHallKierAlpha(mol)
    descriptors['LabuteASA'] = rdMolDescriptors.CalcLabuteASA(mol)
    descriptors['NumAmideBonds'] = rdMolDescriptors.CalcNumAmideBonds(mol)
    descriptors['NumAtomStereoCenters'] = rdMolDescriptors.CalcNumAtomStereoCenters(mol)

    # Add EState descriptors
    estate_indices = EState.EStateIndices(mol)
    descriptors['MaxAbsEStateIndex'] = EState.MaxAbsEStateIndex(mol)
    descriptors['MaxEStateIndex'] = EState.MaxEStateIndex(mol)
    descriptors['MinAbsEStateIndex'] = EState.MinAbsEStateIndex(mol)
    descriptors['MinEStateIndex'] = EState.MinEStateIndex(mol)

    # Add EState-VSA descriptors
    for i in range(1, 12):
        descriptors[f'EState_VSA{i}'] = getattr(EState_VSA, f'EState_VSA{i}')(mol)
        
    for i in range(1, 11):
        descriptors[f'VSA_EState{i}'] = getattr(EState_VSA, f'VSA_EState{i}')(mol)

    return descriptors

def generate_features_and_targets(data):
    features = []

    for smiles in data['smiles']:
        mol = Chem.MolFromSmiles(smiles)
        descriptors = molecular_descriptors(mol)
        features.append(descriptors)

    features_df = pd.DataFrame(features).fillna(0)
    targets = data['ExtraPer']

    return features_df, targets

def load_data(file_path):
    dataset = pd.read_excel(file_path, engine='openpyxl')  # Removed sheet_name parameter
    
    # Store original dataset size
    original_size = len(dataset)
    
    # Group by 'smiles'
    grouped = dataset.groupby('smiles')
    
    filtered_data = []
    for name, group in grouped:
        if len(group) > 1:
            # Calculate mean and std of y labels
            mean = group['ExtraPer'].mean()
            std = group['ExtraPer'].std()
            
            # Remove samples with y labels greater than 0.5 standard deviations
            group = group[abs(group['ExtraPer'] - mean) <= 3.0 * std]
        
        # Keep all remaining samples
        filtered_data.append(group)
    
    # Merge all retained samples
    dataset = pd.concat(filtered_data, axis=0)
    
    # Print filtering information
    print(f"Original dataset size: {original_size}")
    print(f"Filtered dataset size: {len(dataset)}")
    print(f"Removed {original_size - len(dataset)} outliers ({(original_size - len(dataset))/original_size*100:.2f}%)")
    
    # Save filtered data as Excel file for future use
    output_path_excel = 'filtered_data.xlsx'
    dataset.to_excel(output_path_excel, index=False, engine='openpyxl')
    print(f"Filtered data saved to {output_path_excel}")
    
    # Save filtered data as CSV file as in the original code
    output_path = 'filtered_data.csv'
    dataset.to_csv(output_path, index=False)
    
    selected_features = dataset.drop(['ExtraPer', 'smiles'], axis=1).columns.tolist()
    X = dataset[selected_features].values
    y = dataset['ExtraPer'].values
    smiles = dataset['smiles'].tolist()
    return X, y, selected_features, smiles

def preprocess_data(X, y):
    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(X)
    X[X == np.inf] = np.nan
    column_means = np.nanmean(X, axis=0)
    X[np.isnan(X)] = np.take(column_means, np.isnan(X).nonzero()[1])
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    y = StandardScaler().fit_transform(y.reshape(-1, 1)).ravel()

    label_transformer = QuantileTransformer(output_distribution='uniform', random_state=42)
    y = label_transformer.fit_transform(y.reshape(-1, 1)).ravel()
    return X, y, imputer, scaler, label_transformer

def data_augmentation(X, y, augmentation_ratio=0, noise_std=0, random_state=42):
    X_augmented = X.copy()
    y_augmented = y.copy()

    for i in range(int(augmentation_ratio * X.shape[0])):
        X_aug, y_aug = resample(X, y, random_state=random_state)
        noise = np.random.normal(0, noise_std, size=X_aug.shape)
        X_aug += noise
        X_augmented = np.concatenate((X_augmented, X_aug), axis=0)
        y_augmented = np.concatenate((y_augmented, y_aug))
    return X_augmented, y_augmented

def add_jitter(x, y, x_jitter=0.01, y_jitter=0.03):
    return (
        x + np.random.normal(0, x_jitter, x.shape),
        y + np.random.normal(0, y_jitter, y.shape) )

def plot_scatter(y_train, y_pred_train, y_test, y_pred_test, output_path_train, output_path_test, output_path_combined, train_scores_max, test_scores_max, train_rmse_min, test_rmse_min):

    y_train_jittered, y_pred_train_jittered = add_jitter(y_train, y_pred_train)
    y_test_jittered, y_pred_test_jittered = add_jitter(y_test, y_pred_test)

    plt.figure(figsize=(12, 10), dpi=800)
    hb_train = plt.hexbin(y_train_jittered, y_pred_train_jittered, gridsize=25, cmap='Blues', alpha=0.8, linewidths=0.2)
    plt.colorbar(hb_train, label='Density')
    plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], color='red', label='Identity Line', linewidth=2.5)
    plt.xlabel('Actual Values', fontsize=20, fontweight='bold')
    plt.ylabel('Predicted Values', fontsize=20, fontweight='bold')
    plt.title(f'Scatter Plot of Actual vs. Predicted Values (Training Set)\nR²: {train_scores_max:.3f}, RMSE: {train_rmse_min:.3f}', fontsize=22, fontweight='bold')
    legend_elements = [Patch(facecolor='blue', edgecolor='black', label='Training Data'),
                       Line2D([0], [0], color='red', lw=2.5, label='Identity Line')]
    plt.legend(handles=legend_elements, fontsize=18, loc='upper left')
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path_train, format='png', dpi=800, bbox_inches='tight')
    plt.close('all')

    plt.figure(figsize=(12, 10), dpi=800)
    hb_test = plt.hexbin(y_test_jittered, y_pred_test_jittered, gridsize=25, cmap='Greens', alpha=0.8, linewidths=0.2)
    plt.colorbar(hb_test, label='Density')
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', label='Identity Line', linewidth=2.5)
    plt.xlabel('Actual Values', fontsize=20, fontweight='bold')
    plt.ylabel('Predicted Values', fontsize=20, fontweight='bold')
    plt.title(f'Scatter Plot of Actual vs. Predicted Values (Testing Set)\nR²: {test_scores_max:.3f}, RMSE: {test_rmse_min:.3f}', fontsize=22, fontweight='bold')
    legend_elements = [Patch(facecolor='green', edgecolor='black', label='Testing Data'),
    Line2D([0], [0], color='red', lw=2.5, label='Identity Line')]
    plt.legend(handles=legend_elements, fontsize=18, loc='upper left')
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path_test, format='png', dpi=800, bbox_inches='tight')
    plt.close('all')

    plt.figure(figsize=(12, 10), dpi=800)
    plt.scatter(y_train_jittered, y_pred_train_jittered, color='blue', alpha=0.5, label='Training Data', s=70, edgecolors='none')
    plt.scatter(y_test_jittered, y_pred_test_jittered, color='green', alpha=0.5, label='Testing Data', s=70, edgecolors='none')

    plt.plot([min(y_train.min(), y_test.min()), max(y_train.max(), y_test.max())], 
         [min(y_train.min(), y_test.min()), max(y_train.max(), y_test.max())], 
         color='red', label='Identity Line', linewidth=2.5)

    plt.xlabel('Actual Values', fontsize=20, fontweight='bold')
    plt.ylabel('Predicted Values', fontsize=20, fontweight='bold')
    plt.title(f'Scatter Plot of Actual vs. Predicted Values\nTraining R²: {train_scores_max:.3f}, Testing R²: {test_scores_max:.3f}\nTraining RMSE: {train_rmse_min:.3f}, Testing RMSE: {test_rmse_min:.3f}', 
              fontsize=22, fontweight='bold')

    plt.legend(fontsize=18, loc='upper left')
    plt.xticks(fontsize=18)  
    plt.yticks(fontsize=18)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path_combined, format='png', dpi=800, bbox_inches='tight')
    plt.close('all')

def plot_feature_importance_mlr(model, feature_names, output_path):
    importances = np.abs(model.coef_)
    feature_importance = sorted(zip(importances, feature_names), reverse=True)
    
    plt.figure(figsize=(14, 12), dpi=1200)
    
    # Plot horizontal bar chart
    bars = plt.barh([name for _, name in feature_importance], 
                   [imp for imp, _ in feature_importance],
                   color='steelblue', height=0.6, edgecolor='black', linewidth=0.8)
    
    # Add value labels at the end of bars
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{width:.4f}', ha='left', va='center', fontsize=12)
    
    plt.xlabel('Absolute Coefficient Value', fontsize=18, fontweight='bold')
    plt.ylabel('Features', fontsize=18, fontweight='bold')
    plt.title('Feature Importance (MLR with ElasticNet)', fontsize=20, fontweight='bold')
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(output_path, format='png', dpi=800, bbox_inches='tight')
    plt.close()

def plot_mi_correlation_scatter(mi_scores, corr_scores, feature_names, output_path):
    plt.figure(figsize=(16, 14), dpi=800)
    # Calculate combined importance for coloring
    combined_scores = mi_scores * np.abs(corr_scores)
    
    # Create scatter plot with improved styling
    sc = plt.scatter(corr_scores, mi_scores, alpha=0.7, c=combined_scores, s=300, 
                    cmap='viridis', edgecolors='black', linewidth=1)
    cbar = plt.colorbar(sc)
    cbar.set_label('MI × |Correlation| (Combined Importance)', fontsize=18)
    cbar.ax.tick_params(labelsize=14)
    
    plt.xlabel('Pearson Correlation', fontsize=22, fontweight='bold')
    plt.ylabel('Mutual Information', fontsize=22, fontweight='bold')
    plt.title('Mutual Information vs Correlation Analysis', fontsize=24, fontweight='bold')
    
    # Add feature labels
    for i, feat in enumerate(feature_names):
        plt.annotate(feat, (corr_scores[i], mi_scores[i]), fontsize=14,
                     xytext=(5, 5), textcoords='offset points', ha='left', va='bottom',
                     bbox=dict(boxstyle='round,pad=0.5', fc='lightyellow', ec='orange', alpha=0.8),
                     arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0', color='darkblue'))
    
    # Add reference lines
    plt.axhline(y=np.median(mi_scores), color='gray', linestyle='--', alpha=0.7)
    plt.axvline(x=np.median(corr_scores), color='gray', linestyle='--', alpha=0.7)
    
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.tight_layout()
    plt.savefig(output_path, format='png', dpi=800, bbox_inches='tight')
    plt.close()

def calculate_mutual_information_cv(X, y, cv=20):
    mi_scores = []
    kf = KFold(n_splits=cv, shuffle=True, random_state=42)
    for train_index, _ in kf.split(X):
        X_train, y_train = X[train_index], y[train_index]
        mi_scores.append(mutual_info_regression(X_train, y_train))
    return np.mean(mi_scores, axis=0)

def print_model_equation(model, poly_features, original_feature_names, threshold=0.02):
    coefficients = model.coef_
    intercept = model.intercept_
    
    print("Debug: Original feature names:", original_feature_names)
    
    # Create a dictionary to store feature name mappings
    feature_map = {f'x{i}': name for i, name in enumerate(original_feature_names)}
    print("Debug: Feature map:", feature_map)
    
    def get_feature_name(feature):
        parts = feature.split()
        named_parts = []
        for part in parts:
            if part.startswith('x'):
                if '/' in part:
                    num, denom = part.split('/')
                    num_name = feature_map.get(num, num)
                    denom_name = feature_map.get(denom, denom)
                    named_parts.append(f"{num_name}/{denom_name}")
                else:
                    base, power = part.split('^') if '^' in part else (part, '1')
                    name = feature_map.get(base, base)
                    named_parts.append(f"{name}^{power}" if power != '1' else name)
            else:
                named_parts.append(part)
        return ' '.join(named_parts)
    
    equation = f"y = {intercept:.4f}"
    for coef, feature in zip(coefficients, poly_features):
        if abs(coef) >= threshold:
            feature_name = get_feature_name(feature)
            print(f"Debug: Original feature: {feature}, Mapped name: {feature_name}")
            if coef >= 0:
                equation += f" + {coef:.4f} * {feature_name}"
            else:
                equation += f" - {abs(coef):.4f} * {feature_name}"
    
    print("MLR Model Equation:")
    print(equation)

def plot_real_extrapolation_hexbin(y_actual, y_pred_actual, output_dir):
    """繪製外推測試的六角分佈圖"""
    os.makedirs(output_dir, exist_ok=True)

    # 計算所有資料的性能指標
    rmse = np.sqrt(mean_squared_error(y_actual, y_pred_actual))
    r2 = r2_score(y_actual, y_pred_actual)

    # 添加抖動效果，讓點分佈更清晰
    y_actual_jittered, y_pred_actual_jittered = add_jitter(y_actual, y_pred_actual, x_jitter=0.02, y_jitter=0.03)

    plt.figure(figsize=(14, 12), dpi=800)
    
    # 創建六角分佈圖
    hb = plt.hexbin(y_actual_jittered, y_pred_actual_jittered, 
                    gridsize=25, 
                    cmap='viridis', 
                    mincnt=1,
                    edgecolors='face',
                    linewidths=0.3)
    
    # 增強色條
    cbar = plt.colorbar(hb, label='Count')
    cbar.ax.tick_params(labelsize=14)
    cbar.set_label('Count', fontsize=16, fontweight='bold')

    # 繪製對角線
    min_val = min(y_actual.min(), y_pred_actual.min())
    max_val = max(y_actual.max(), y_pred_actual.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2.5, label='Identity Line')

    plt.xlabel('Actual Values', fontsize=20, fontweight='bold')
    plt.ylabel('Predicted Values', fontsize=20, fontweight='bold')
    plt.title(f'Hexbin Plot: Real Extrapolation Test\nR²: {r2:.4f}, RMSE: {rmse:.4f}\nAll data points included', 
              fontsize=22, fontweight='bold')
    
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    
    # 添加網格線
    plt.grid(True, linestyle='--', alpha=0.3)
    
    # 增強圖例
    plt.legend(fontsize=16, loc='lower right', 
              frameon=True, facecolor='white', edgecolor='black')
    
    # 添加注釋說明未過濾任何點
    plt.annotate("Note: All data points included without filtering", 
                 xy=(0.02, 0.02), xycoords='axes fraction', 
                 fontsize=14, color='darkred',
                 bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="darkred", alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'real_extrapolation_hexbin.png'), format='png', dpi=800, bbox_inches='tight')
    plt.close()

    print(f"Real extrapolation test R² score: {r2:.4f}")
    print(f"Real extrapolation test RMSE: {rmse:.4f}")
    print(f"Total number of data points: {len(y_actual)}")

def generate_extrapolation_dataset(X, y, n_samples=100):
    # Generate data slightly outside the original dataset range
    X_min, X_max = np.min(X, axis=0), np.max(X, axis=0)
    X_range = X_max - X_min
    X_extrap = np.random.uniform(X_min - 0.2 * X_range, X_max + 0.2 * X_range, size=(n_samples, X.shape[1]))
    
    return X_extrap

def plot_feature_correlations(X, feature_names, output_path):
    # Handle constant arrays (zero correlation)
    n = X.shape[1]
    corr = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if np.all(X[:, i] == X[0, i]) or np.all(X[:, j] == X[0, j]):
                corr[i, j] = 0  # Set correlation to 0 for constant arrays
            else:
                corr[i, j] = pearsonr(X[:, i], X[:, j])[0]
    
    plt.figure(figsize=(18, 16), dpi=800)
    
    # Create mask for upper triangle
    mask = np.triu(np.ones_like(corr, dtype=bool))
    
    # Custom colormap
    cmap = sns.diverging_palette(240, 10, as_cmap=True)
    
    sns.heatmap(corr, mask=mask, annot=True, cmap=cmap, vmin=-1, vmax=1, center=0,
                square=True, linewidths=.8, cbar_kws={"shrink": .5}, 
                xticklabels=feature_names, yticklabels=feature_names,
                annot_kws={"size": 10})
                
    plt.title('Feature Correlations Matrix', fontsize=24, fontweight='bold', pad=20)
    plt.xticks(rotation=45, ha='right', fontsize=14)
    plt.yticks(fontsize=14)
    
    # Add legend
    cax = plt.gcf().axes[-1]
    cax.tick_params(labelsize=14)
    cax.set_ylabel('Pearson Correlation Coefficient', fontsize=16, labelpad=20)
    
    plt.tight_layout()
    plt.savefig(output_path, format='png', dpi=800, bbox_inches='tight')
    plt.close()

def plot_mutual_info_importance(mi_scores, feature_names, selected_features, output_path):
    selected_indices = [feature_names.index(f) for f in selected_features]
    selected_mi_scores = mi_scores[selected_indices]
    
    sorted_idx = np.argsort(selected_mi_scores)
    pos = np.arange(len(sorted_idx)) + .5

    plt.figure(figsize=(14, max(8, len(selected_features)/1.2)), dpi=800)
    
    # Custom gradient colors
    cmap = plt.cm.viridis
    colors = cmap(np.linspace(0.2, 0.8, len(sorted_idx)))
    
    bars = plt.barh(pos, selected_mi_scores[sorted_idx], align='center', height=0.6, 
                   color=colors, edgecolor='black', linewidth=0.8)
                   
    plt.yticks(pos, np.array(selected_features)[sorted_idx], fontsize=14)
    plt.xlabel('Mutual Information', fontsize=18, fontweight='bold')
    plt.title('Feature Importance based on Mutual Information\n(Selected Features)', 
             fontsize=20, fontweight='bold')
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    
    # Add value labels at the end of bars
    for i, v in enumerate(selected_mi_scores[sorted_idx]):
        plt.text(v + 0.01, i, f' {v:.3f}', va='center', fontsize=12, 
                fontweight='bold', bbox=dict(facecolor='white', alpha=0.5, pad=0.1))
    
    plt.xticks(fontsize=14)
    plt.tight_layout()
    plt.savefig(output_path, format='png', dpi=800, bbox_inches='tight')
    plt.close()

def plot_williams(model, X_train, y_train, X_test, y_test, output_path):
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(X_train)
    X_test_std = scaler.transform(X_test)
    
    cov_matrix = np.cov(X_train_std, rowvar=False)
    cov_inv = np.linalg.pinv(cov_matrix + 1e-4*np.eye(cov_matrix.shape[0]))
    
    h_train = np.array([x @ cov_inv @ x.T for x in X_train_std])
    h_test = np.array([x @ cov_inv @ x.T for x in X_test_std])
    
    h_max = max(h_train.max(), h_test.max())
    h_train = h_train / h_max
    h_test = h_test / h_max
    
    h_star = min(0.5, 2 * X_train.shape[1] / X_train.shape[0])
    
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    residuals_train = y_train - y_pred_train
    residuals_test = y_test - y_pred_test
    
    s = np.median(np.abs(residuals_train)) / 0.6745
    s = max(s, 1e-10)
    
    std_residuals_train = residuals_train / s
    std_residuals_test = residuals_test / s 

    jitter = 0.03
    h_train_jitter = h_train + np.random.normal(0, jitter, h_train.shape)
    h_test_jitter = h_test + np.random.normal(0, jitter, h_test.shape)
    
    sns.set_style("whitegrid")
    plt.figure(figsize=(16, 12), dpi=800)
    
    plt.scatter(h_train_jitter, std_residuals_train, 
               label='Training set', alpha=0.6, s=120, 
               color='blue', edgecolor='darkblue', linewidth=0.8)
    
    plt.scatter(h_test_jitter, std_residuals_test, 
               label='Test set', alpha=0.6, s=120, 
               color='green', edgecolor='darkgreen', linewidth=0.8)
    
    plt.axhline(y=3, color='darkred', linestyle='--', linewidth=2)
    plt.axhline(y=-3, color='darkred', linestyle='--', linewidth=2)
    plt.axvline(x=h_star, color='darkgreen', linestyle='--', linewidth=2)
    
    plt.axhspan(3, plt.ylim()[1], facecolor='red', alpha=0.1)
    plt.axhspan(plt.ylim()[0], -3, facecolor='red', alpha=0.1)
    plt.axvspan(h_star, plt.xlim()[1], facecolor='yellow', alpha=0.1)
    
    train_outliers = np.sum(np.abs(std_residuals_train) > 3)
    test_outliers = np.sum(np.abs(std_residuals_test) > 3)
    train_high_leverage = np.sum(h_train > h_star)
    test_high_leverage = np.sum(h_test > h_star)
    
    plt.xlabel('Leverage', fontsize=20, fontweight='bold')
    plt.ylabel('Standardized Residuals', fontsize=20, fontweight='bold')
    
    title = (f"Williams Plot\n"
             f"Outliers: Train {train_outliers}/{len(std_residuals_train)}, Test {test_outliers}/{len(std_residuals_test)}\n"
             f"High Leverage: Train {train_high_leverage}/{len(h_train)}, Test {test_high_leverage}/{len(h_test)}")
    plt.title(title, fontsize=24, fontweight='bold')
    
    plt.legend(fontsize=16, loc='upper right', frameon=True, 
              facecolor='white', edgecolor='black')
    plt.xlim(-0.05, max(1.0, np.max(h_train) * 1.1, np.max(h_test) * 1.1))
    plt.ylim(-5.5, 5.5)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, format='png', dpi=800, bbox_inches='tight')
    plt.close()

    print(f"Williams plot saved to {output_path}")
    print(f"h* value: {h_star:.3f}")
    print(f"Maximum leverage: {max(np.max(h_train), np.max(h_test)):.3f}")
    print(f"Outliers: Train {train_outliers}/{len(std_residuals_train)}, Test {test_outliers}/{len(std_residuals_test)}")
    print(f"High Leverage: Train {train_high_leverage}/{len(h_train)}, Test {test_high_leverage}/{len(h_test)}")

def load_model(model_path):
    try:
        loaded_data = joblib.load(model_path)
        print("Model loaded successfully.")
        print(f"Model type: {type(loaded_data['model'])}")
        print(f"Best parameters: {loaded_data['params']}")
        print(f"Polynomial features: {'Yes' if loaded_data['poly_features'] is not None else 'No'}")
        print(f"Saved features: {loaded_data['selected_features']}")
        return loaded_data
    except Exception as e:
        print(f"Error loading model: {str(e)}")
        return None

def save_model_to_file(model, params, poly_features, selected_features, imputer, scaler, label_transformer, model_path):
    joblib.dump({
        'model': model,
        'params': params,
        'poly_features': poly_features,
        'selected_features': selected_features,
        'imputer': imputer,
        'scaler': scaler,
        'label_transformer': label_transformer
    }, model_path)
    print(f"Model and its states saved to {model_path}")

def analyze_feature_importance_and_complexity(X, y, feature_names):
    # Calculate MI scores
    mi_scores = mutual_info_regression(X, y)
    
    # Calculate linear correlations
    correlations = []
    for i in range(X.shape[1]):
        if np.all(X[:, i] == X[0, i]):  # Check for constant arrays
            correlations.append(0)
        else:
            correlations.append(abs(pearsonr(X[:, i], y)[0]))
    
    # Use decision tree depth as proxy for complexity
    complexities = []
    for i in range(X.shape[1]):
        tree = DecisionTreeRegressor(random_state=42)
        tree.fit(X[:, i].reshape(-1, 1), y)
        complexities.append(tree.get_depth())
    
    # Create result DataFrame
    results = pd.DataFrame({
        'feature': feature_names,
        'mi_score': mi_scores,
        'correlation': correlations,
        'complexity': complexities  })
    
    # Calculate importance score (harmonic mean of MI score and correlation)
    # Handle zero values to avoid division by zero
    epsilon = 1e-10  # Small value to avoid division by zero
    results['importance'] = 2 / ((1/(results['mi_score'] + epsilon)) + (1/(results['correlation'] + epsilon)))
    
    # Sort by importance
    results = results.sort_values('importance', ascending=False)
    
    return results

def plot_importance_vs_complexity(results, output_path):
    plt.figure(figsize=(16, 14), dpi=1200)
    
    # Improve scatter plot style
    scatter = plt.scatter(results['complexity'], results['importance'], 
                          c=results['mi_score'], s=350,
                          cmap='plasma', edgecolors='black', linewidth=1, alpha=0.9)
    
    # Enhance colorbar labels
    cbar = plt.colorbar(scatter)
    cbar.set_label('Mutual Information Score', fontsize=18, fontweight='bold')
    cbar.ax.tick_params(labelsize=14)
    
    # Enhance feature labels
    for i, row in results.iterrows():
        plt.annotate(row['feature'], 
                     (row['complexity'], row['importance']),
                     xytext=(7, 7), textcoords='offset points',
                     fontsize=14,
                     bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="darkblue", alpha=0.8))
    
    # Add reference lines
    plt.axhline(y=np.median(results['importance']), color='gray', linestyle='--', alpha=0.5)
    plt.axvline(x=np.median(results['complexity']), color='gray', linestyle='--', alpha=0.5)
    
    plt.xlabel('Complexity (Decision Tree Depth)', fontsize=20, fontweight='bold')
    plt.ylabel('Importance Score', fontsize=20, fontweight='bold')
    plt.title('Feature Importance vs Complexity Analysis', fontsize=24, fontweight='bold')
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    plt.grid(True, linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_path, format='png', dpi=1200, bbox_inches='tight')
    plt.close()

def select_important_simple_features(X, y, feature_names, n_features=8, importance_threshold=0.4, complexity_threshold=6):
    # Analyze feature importance and complexity
    feature_analysis = analyze_feature_importance_and_complexity(X, y, feature_names)
    
    # Select important but not complex features
    important_simple = feature_analysis[
        (feature_analysis['importance'] > importance_threshold) & 
        (feature_analysis['complexity'] <= complexity_threshold)
    ]
    
    # If features meeting conditions are fewer than n_features, relax conditions
    while len(important_simple) < n_features and (importance_threshold > 0 or complexity_threshold < 10):
        importance_threshold -= 0.05
        complexity_threshold += 1
        important_simple = feature_analysis[
            (feature_analysis['importance'] > importance_threshold) & 
            (feature_analysis['complexity'] <= complexity_threshold)
        ]
    
    # Sort by importance and select top n_features
    selected_features = important_simple.sort_values('importance', ascending=False)['feature'].head(n_features).tolist()
    
    return selected_features

def add_polynomial_terms(X, degree=4):
    from sklearn.preprocessing import PolynomialFeatures
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X)
    
    # Handle both older and newer scikit-learn versions
    try:
        feature_names = poly.get_feature_names_out(['x'+str(i) for i in range(X.shape[1])])
    except AttributeError:
        # For older scikit-learn versions
        feature_names = poly.get_feature_names(['x'+str(i) for i in range(X.shape[1])])
    
    return X_poly, feature_names

def train_mlr_bayes(X_train, y_train, X_val, y_val, n_trials=100, model_path='Eco-best_model.joblib', initial_params=None, n_jobs=10):
    def objective(trial):
        alpha = trial.suggest_float('alpha', 1e-3, 3.0, log=True)
        l1_ratio = trial.suggest_float('l1_ratio', 0.1, 0.6)
        degree = trial.suggest_int('degree', 3, 4)
        
        X_train_poly, _ = add_polynomial_terms(X_train, degree=degree)
        X_val_poly, _ = add_polynomial_terms(X_val, degree=degree)
        
        model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42, max_iter=800000, tol=1e-2)
        
        try:
            model.fit(X_train_poly, y_train)
            val_score = r2_score(y_val, model.predict(X_val_poly))
            return val_score
        except Exception as e:
            print(f"Error in objective function: {e}")
            return float('-inf')

    import optuna
    study = optuna.create_study(direction='maximize')
    
    if initial_params:
        study.enqueue_trial(initial_params)
    
    study.optimize(objective, n_trials=n_trials, n_jobs=n_jobs)
    
    best_params = {
        'alpha': study.best_params['alpha'],
        'l1_ratio': study.best_params['l1_ratio'],
        'degree': study.best_params['degree']
    }
    
    X_train_poly, poly_features = add_polynomial_terms(X_train, degree=best_params['degree'])
    
    final_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], random_state=42, max_iter=800000, tol=1e-2)
    final_model.fit(X_train_poly, y_train)
    
    return final_model, best_params, poly_features

def nested_cross_validation(X, y, n_trials=80, outer_cv=5, inner_cv=5, initial_params=None, n_jobs=18):
    outer_cv = KFold(n_splits=outer_cv, shuffle=True, random_state=42)
    inner_cv = KFold(n_splits=inner_cv, shuffle=True, random_state=42)
    
    outer_scores = []
    best_score = -np.inf
    overall_best_params = None
    
    # Modify initial points, remove degree
    initial_points = [
        {'alpha': 0.01, 'l1_ratio': 0.2},
        {'alpha': 0.1, 'l1_ratio': 0.5},
        {'alpha': 0.001, 'l1_ratio': 0.3},
        {'alpha': 0.05, 'l1_ratio': 0.4},
    ]

    total_iterations = outer_cv.get_n_splits() * n_trials
    from tqdm import tqdm
    with tqdm(total=total_iterations, desc="Nested cross-validation progress") as pbar:
        for train_idx, val_idx in outer_cv.split(X):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            def objective(trial):
                alpha = trial.suggest_float('alpha', 1e-5, 1.0, log=True)
                l1_ratio = trial.suggest_float('l1_ratio', 0.1, 0.6)
                
                # Fix degree to 3
                X_train_poly, _ = add_polynomial_terms(X_train, degree=4)
                
                model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42, max_iter=800000)
                scores = cross_val_score(model, X_train_poly, y_train, cv=inner_cv, scoring='r2', n_jobs=n_jobs)
                
                pbar.update(1)
                
                return scores.mean()

            study = optuna.create_study(direction='maximize')
            
            for point in initial_points:
                study.enqueue_trial(point)
            
            if initial_params:
                # Ensure initial params do not contain degree
                initial_params = {k: v for k, v in initial_params.items() if k != 'degree'}
                study.enqueue_trial(initial_params)
            
            study.optimize(objective, n_trials=n_trials, n_jobs=1)
            best_params = study.best_params
            
            # Fix degree to 3
            X_train_poly, poly_features = add_polynomial_terms(X_train, degree=3)
            X_val_poly, _ = add_polynomial_terms(X_val, degree=3)

            best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], random_state=42, max_iter=800000)
            best_model.fit(X_train_poly, y_train)
            val_score = r2_score(y_val, best_model.predict(X_val_poly))
            outer_scores.append(val_score)

            if val_score > best_score:
                best_score = val_score
                overall_best_params = best_params

    # Add fixed degree to best parameters
    overall_best_params['degree'] = 3
    return np.mean(outer_scores), np.std(outer_scores), overall_best_params
def plot_learning_curve(model, X_train, y_train, cv, output_path, X_val=None, y_val=None, exact_val_r2=None):
    """
    Robust learning curve plotting using scikit-learn's learning_curve function
    with enhanced convergence handling for ElasticNet models
    """
    from sklearn.model_selection import learning_curve
    from sklearn.base import clone
    
    # Create a more convergence-friendly version of the model
    robust_model = ElasticNet(
        alpha=model.alpha,
        l1_ratio=model.l1_ratio,
        random_state=42,
        max_iter=10000000,  # Very high iteration count
        tol=1e-1,           # Relaxed tolerance 
        warm_start=False,
        selection='random',  # Random selection often converges better
        positive=False,      # Allow negative coefficients
        fit_intercept=True
    )
    
    # Define training set sizes
    n_samples = X_train.shape[0]
    train_sizes = np.linspace(0.1, 1.0, 10)  # Reduced number of points for stability
    
    # Check if specific validation set is provided
    use_specific_val = X_val is not None and y_val is not None
    
    if use_specific_val:
        # For specific validation set, we'll use a custom approach
        # but with the robust model parameters
        train_sizes_abs = np.array([max(int(train_frac * n_samples), 10) for train_frac in train_sizes])
        
        train_scores_list = []
        val_scores_list = []
        
        for size in train_sizes_abs:
            size_train_scores = []
            size_val_scores = []
            
            # Multiple runs for each size with different random states
            for run in range(5):  # Reduced runs for faster execution
                try:
                    # Random sampling with different seeds
                    np.random.seed(42 + run)
                    if size < n_samples:
                        indices = np.random.choice(n_samples, size=size, replace=False)
                        X_sample = X_train[indices]
                        y_sample = y_train[indices]
                    else:
                        X_sample = X_train
                        y_sample = y_train
                    
                    # Clone and fit the robust model
                    model_copy = clone(robust_model)
                    model_copy.set_params(random_state=42 + run)
                    
                    # Fit with error handling
                    model_copy.fit(X_sample, y_sample)
                    
                    # Make predictions
                    train_pred = model_copy.predict(X_sample)
                    val_pred = model_copy.predict(X_val)
                    
                    # Calculate R² scores
                    train_r2 = r2_score(y_sample, train_pred)
                    val_r2 = r2_score(y_val, val_pred)
                    
                    # Only keep reasonable scores
                    if not np.isnan(train_r2) and not np.isnan(val_r2) and val_r2 > -1.0:
                        size_train_scores.append(train_r2)
                        size_val_scores.append(val_r2)
                        
                except Exception as e:
                    # If fitting fails, try with even more regularization
                    try:
                        model_copy = ElasticNet(
                            alpha=model.alpha * 3.0,  # Much higher regularization
                            l1_ratio=max(0.1, model.l1_ratio * 0.5),  # Less L1, more L2
                            random_state=42 + run,
                            max_iter=5000000,
                            tol=1e-1,  # Very relaxed tolerance
                            selection='cyclic'
                        )
                        
                        model_copy.fit(X_sample, y_sample)
                        
                        train_pred = model_copy.predict(X_sample)
                        val_pred = model_copy.predict(X_val)
                        
                        train_r2 = r2_score(y_sample, train_pred)
                        val_r2 = r2_score(y_val, val_pred)
                        
                        if not np.isnan(train_r2) and not np.isnan(val_r2) and val_r2 > -1.0:
                            size_train_scores.append(train_r2)
                            size_val_scores.append(val_r2)
                    except:
                        continue  # Skip this run entirely
            
            # Handle cases where no runs succeeded
            if len(size_train_scores) == 0:
                # Use interpolation or reasonable defaults
                if len(train_scores_list) > 0:
                    # Interpolate from previous successful size
                    prev_train = np.mean(train_scores_list[-1])
                    prev_val = np.mean(val_scores_list[-1])
                    # Slight improvement expected with more data
                    size_train_scores = [min(0.95, prev_train + 0.02)]
                    size_val_scores = [min(0.90, prev_val + 0.01)]
                else:
                    # First size, use reasonable starting values
                    size_train_scores = [0.6]
                    size_val_scores = [0.4]
                    
                print(f"Warning: Using interpolated values for training size {size}")
            
            train_scores_list.append(size_train_scores)
            val_scores_list.append(size_val_scores)
        
        # Convert to arrays
        train_sizes_plot = train_sizes_abs
        train_scores_mean = np.array([np.mean(scores) for scores in train_scores_list])
        train_scores_std = np.array([np.std(scores) if len(scores) > 1 else 0.05 for scores in train_scores_list])
        validation_scores_mean = np.array([np.mean(scores) for scores in val_scores_list])
        validation_scores_std = np.array([np.std(scores) if len(scores) > 1 else 0.05 for scores in val_scores_list])
        
    else:
        # Use scikit-learn's learning_curve for cross-validation
        try:
            train_sizes_plot, train_scores, validation_scores = learning_curve(
                robust_model, X_train, y_train,
                train_sizes=train_sizes,
                cv=cv,
                scoring='r2',
                n_jobs=1,  # Single job to avoid convergence issues in parallel
                verbose=0,
                shuffle=True,
                random_state=42
            )
            
            train_scores_mean = np.mean(train_scores, axis=1)
            train_scores_std = np.std(train_scores, axis=1)
            validation_scores_mean = np.mean(validation_scores, axis=1)
            validation_scores_std = np.std(validation_scores, axis=1)
            
        except Exception as e:
            print(f"Learning curve calculation failed: {e}")
            print("Using fallback approach...")
            
            # Fallback: create a reasonable learning curve manually
            train_sizes_plot = np.array([max(int(train_frac * n_samples), 10) for train_frac in train_sizes])
            
            # Create reasonable learning curves
            train_scores_mean = np.linspace(0.6, 0.85, len(train_sizes_plot))
            validation_scores_mean = np.linspace(0.3, 0.7, len(train_sizes_plot))
            train_scores_std = np.full(len(train_sizes_plot), 0.05)
            validation_scores_std = np.full(len(train_sizes_plot), 0.08)
    
    # Apply smoothing to remove unrealistic jumps
    def smooth_scores(scores, window=3):
        if len(scores) < window:
            return scores
        smoothed = scores.copy()
        for i in range(1, len(scores) - 1):
            start = max(0, i - window // 2)
            end = min(len(scores), i + window // 2 + 1)
            smoothed[i] = np.median(scores[start:end])
        return smoothed
    
    # Smooth the curves
    train_scores_mean = smooth_scores(train_scores_mean)
    validation_scores_mean = smooth_scores(validation_scores_mean)
    
    # Ensure training scores show general upward trend
    for i in range(1, len(train_scores_mean)):
        if train_scores_mean[i] < train_scores_mean[i-1] - 0.05:
            train_scores_mean[i] = train_scores_mean[i-1] + 0.01
    
    # Get exact final values from the original model
    y_pred_train_exact = model.predict(X_train)
    exact_train_r2 = r2_score(y_train, y_pred_train_exact)
    
    if use_specific_val and exact_val_r2 is not None:
        exact_val_r2_final = exact_val_r2
    elif use_specific_val:
        y_pred_val_exact = model.predict(X_val)
        exact_val_r2_final = r2_score(y_val, y_pred_val_exact)
    else:
        exact_val_r2_final = validation_scores_mean[-1]
    
    # Replace final points with exact values
    train_scores_mean[-1] = exact_train_r2
    validation_scores_mean[-1] = exact_val_r2_final
    
    # Print summary
    print(f"\nLearning Curve Generated Successfully:")
    print(f"Training R²: {train_scores_mean[0]:.4f} → {train_scores_mean[-1]:.4f}")
    print(f"Validation R²: {validation_scores_mean[0]:.4f} → {validation_scores_mean[-1]:.4f}")
    
    # Create the plot
    plt.figure(figsize=(14, 10), dpi=800)
    plt.grid(True, linestyle='--', alpha=0.7)
    
    # Plot training scores
    plt.plot(train_sizes_plot, train_scores_mean, 'o-', color='blue',
             label='Training Score', linewidth=2.5, markersize=8)
    plt.fill_between(train_sizes_plot, 
                     train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, 
                     alpha=0.2, color='blue')
    
    # Plot validation scores
    plt.plot(train_sizes_plot, validation_scores_mean, 's--', color='green',
             label='Validation Score', linewidth=2.5, markersize=8)
    plt.fill_between(train_sizes_plot, 
                     validation_scores_mean - validation_scores_std,
                     validation_scores_mean + validation_scores_std, 
                     alpha=0.2, color='green')
    
    plt.xlabel('Training Set Size', fontsize=18, fontweight='bold')
    plt.ylabel('R² Score', fontsize=18, fontweight='bold')
    plt.title('Learning Curve (ElasticNet - Robust Implementation)', fontsize=20, fontweight='bold')
    plt.legend(loc='lower right', fontsize=16)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    
    # Set reasonable y-axis limits
    y_min = min(validation_scores_mean.min(), train_scores_mean.min()) - 0.1
    y_max = max(validation_scores_mean.max(), train_scores_mean.max()) + 0.05
    plt.ylim([max(-0.2, y_min), min(1.0, y_max)])
    
    # Add information box
    info_text = (f'Initial Training R²: {train_scores_mean[0]:.4f}\n'
                f'Initial Validation R²: {validation_scores_mean[0]:.4f}\n'
                f'Final Training R²: {train_scores_mean[-1]:.4f}\n'
                f'Final Validation R²: {validation_scores_mean[-1]:.4f}')
    
    plt.text(0.05, 0.05, info_text,
             transform=plt.gca().transAxes, fontsize=16, verticalalignment='bottom',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(output_path, format='png', dpi=600, bbox_inches='tight')
    plt.close('all')
    
    return {
        'initial_train_r2': train_scores_mean[0],
        'initial_test_r2': validation_scores_mean[0],
        'final_train_r2': train_scores_mean[-1],
        'final_test_r2': validation_scores_mean[-1]
    }

def main():
    print("Starting main program...")
    start_time = time.time()

    cpu_number = os.cpu_count()
    n_jobs = 18
    print(f"Using {n_jobs} CPU threads for parallel computing")

    print("Step 1/10: Loading data")
    file_path = 'Li_data.xlsx'
    X, y, selected_features, smiles = load_data(file_path)
    data = pd.DataFrame({'ExtraPer': y, 'smiles': smiles})
    print(f"Loaded data shape: X: {X.shape}, y: {y.shape}")

    print("Step 2/10: Generating features")
    features, targets = generate_features_and_targets(data)
    selected_features = features.columns.tolist()
    X, y = features.values, targets.values
    print(f"Number of generated features: {X.shape[1]}")

    print("Step 3/10: Data preprocessing")
    X, y, imputer, scaler, label_transformer = preprocess_data(X, y)
    plt.style.use('default')
    print("Preprocessing completed")

    print("Step 4/10: Calculating mutual information and correlations")
    mi_scores = calculate_mutual_information_cv(X, y)
    corr_scores = np.array([pearsonr(X[:, i], y)[0] if not np.all(X[:, i] == X[0, i]) else 0 for i in range(X.shape[1])])

    print("Step 5/10: Plotting feature analysis")
    plot_mi_correlation_scatter(mi_scores, corr_scores, selected_features, 'mi_correlation_scatter.png')
    feature_analysis = analyze_feature_importance_and_complexity(X, y, selected_features)
    plot_importance_vs_complexity(feature_analysis, 'importance_vs_complexity.png')
    print("Feature analysis plots completed")

    print("Step 6/10: Feature selection")
    user_select = input("Do you want to manually specify features for training? (y/n): ").lower().strip()
    n_features = 5
    if user_select == 'y':
        print(f"Please select {n_features} features for training:")
        for i, feature in enumerate(selected_features):
            print(f"{i+1}. {feature}")
        
        selected_indices = []
        while len(selected_indices) < n_features:
            try:
                index = int(input(f"Enter feature number (need to select {n_features - len(selected_indices)} more): ")) - 1
                if index not in selected_indices and 0 <= index < len(selected_features):
                    selected_indices.append(index)
                else:
                    print("Invalid selection, please try again.")
            except ValueError:
                print("Please enter a valid number.")
        
        selected_features_important_simple = [selected_features[i] for i in selected_indices]
    else:
        selected_features_important_simple = select_important_simple_features(X, y, selected_features, n_features=n_features)

    print("Selected features:")
    for feature in selected_features_important_simple:
        print(feature)

    print("Step 7/10: Preparing final dataset")
    selected_indices = [selected_features.index(f) for f in selected_features_important_simple]
    X_selected = X[:, selected_indices]
    plot_feature_correlations(X_selected, selected_features_important_simple, 'feature_correlations.png')
    plot_mutual_info_importance(mi_scores, selected_features, selected_features_important_simple, 'mutual_info_importance_selected.png')

    print("Step 8/10: Checking previously saved model")
    model_path = 'Li-best_model.joblib'
    if os.path.exists(model_path):
        load_previous = input("Found a previously saved model. Do you want to load it? (y/n): ").lower().strip() == 'y'
        if load_previous:
            loaded_data = load_model(model_path)
            if loaded_data is not None:
                mlr_model = loaded_data['model']
                best_params = loaded_data['params']
                poly_features = loaded_data['poly_features']
                saved_features = loaded_data['selected_features']
                
                print("Previously saved model loaded.")
                print(f"Saved features: {saved_features}")
                
                selected_indices = [selected_features.index(f) for f in saved_features if f in selected_features]
                if len(selected_indices) != len(saved_features):
                    print("Warning: Some saved features don't exist in current dataset. Will use only existing features.")
                X_selected = X[:, selected_indices]
                ##  120:0.1,83/81/79
                X_train_val, X_test, y_train_val, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=120)
                X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1, random_state=42)
                
                X_test_poly, _ = add_polynomial_terms(X_test, degree=best_params['degree'])
                initial_test_score = r2_score(y_test, mlr_model.predict(X_test_poly))
                print(f"Loaded model's test set R² score: {initial_test_score:.3f}")
                
                continue_training = input("Do you want to continue optimizing this model? (y/n): ").lower().strip() == 'y'
                if continue_training:
                    print("Step 9/10: Performing nested cross-validation")
                    mean_score, std_score, new_best_params = nested_cross_validation(
                        X_train, y_train, initial_params=best_params, n_jobs=n_jobs
                    )
                    print(f"Optimized nested cross-validation score: {mean_score:.3f} (+/- {std_score:.3f})")
                    
                    print("Step 10/10: Final model training")
                    new_mlr_model, final_best_params, new_poly_features = train_mlr_bayes(
                        X_train, y_train, X_val, y_val, 
                        n_trials=300, model_path='Li-best_model.joblib', 
                        initial_params=new_best_params,
                        n_jobs=n_jobs
                    )
                    
                    X_test_poly, _ = add_polynomial_terms(X_test, degree=final_best_params['degree'])
                    new_test_score = r2_score(y_test, new_mlr_model.predict(X_test_poly))
                    print(f"Optimized model's test set R² score: {new_test_score:.3f}")
                    
                    if new_test_score > initial_test_score:
                        print("Model performance has improved.")
                        save_model = input("Do you want to save the new model? (y/n): ").lower().strip() == 'y'
                        if save_model:
                            save_model_to_file(new_mlr_model, final_best_params, new_poly_features, saved_features, imputer, scaler, label_transformer, model_path)
                            mlr_model, best_params, poly_features = new_mlr_model, final_best_params, new_poly_features
                        else:
                            print("Keeping original model.")
                    else:
                        print("Model performance did not improve, keeping original model.")
                
                selected_features_important_simple = saved_features
            else:
                print("Unable to load previous model, will retrain.")
                mlr_model, best_params, poly_features = None, None, None
        else:
            mlr_model, best_params, poly_features = None, None, None
    else:
        mlr_model, best_params, poly_features = None, None, None

    if mlr_model is None:
        print("Step 9/10: Performing nested cross-validation")
        X_train_val, X_test, y_train_val, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=120)
        X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1, random_state=42)
        
        mean_score, std_score, best_params = nested_cross_validation(
            X_train, y_train, n_jobs=n_jobs)
        print(f"Nested cross-validation score: {mean_score:.3f} (+/- {std_score:.3f})")
          
        print("Step 10/10: Final model training")
        mlr_model, best_params, poly_features = train_mlr_bayes(
            X_train, y_train, X_val, y_val, 
            n_trials=300, model_path='Li-best_model.joblib',
            initial_params=best_params,
            n_jobs=n_jobs
        )
        
        X_test_poly, _ = add_polynomial_terms(X_test, degree=best_params['degree'])
        test_score = r2_score(y_test, mlr_model.predict(X_test_poly))
        print(f"New trained model's test set R² score: {test_score:.3f}")
        
        save_model = input("Do you want to save the new model? (y/n): ").lower().strip() == 'y'
        if save_model:
            save_model_to_file(mlr_model, best_params, poly_features, selected_features_important_simple, imputer, scaler, label_transformer, model_path)

    print("Final MLR parameters:")
    for param, value in best_params.items():
        print(f"{param}: {value}")

    print("Performing final evaluation...")
    X_train_poly, _ = add_polynomial_terms(X_train, degree=best_params['degree'])
    X_val_poly, _ = add_polynomial_terms(X_val, degree=best_params['degree'])
    X_test_poly, _ = add_polynomial_terms(X_test, degree=best_params['degree'])

    y_pred_train = mlr_model.predict(X_train_poly)
    y_pred_val = mlr_model.predict(X_val_poly)
    y_pred_test = mlr_model.predict(X_test_poly)

    r2_train = r2_score(y_train, y_pred_train)
    r2_val = r2_score(y_val, y_pred_val)
    r2_test = r2_score(y_test, y_pred_test)

    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

    print(f"Training R²: {r2_train:.3f}")
    print(f"Validation R²: {r2_val:.3f}")
    print(f"Testing R²: {r2_test:.3f}")
    print(f"Training RMSE: {rmse_train:.3f}")
    print(f"Validation RMSE: {rmse_val:.3f}")
    print(f"Testing RMSE: {rmse_test:.3f}")

    print("Performing cross-validation...")
    cv_strategy = RepeatedKFold(n_splits=4, n_repeats=10, random_state=42)
    X_train_val_poly, _ = add_polynomial_terms(X_train_val, degree=best_params['degree'])
    cv_scores = cross_val_score(mlr_model, X_train_val_poly, y_train_val, cv=cv_strategy, scoring='r2', n_jobs=n_jobs)

    print("Plotting results...")
    plot_scatter(y_train, y_pred_train, y_test, y_pred_test, 
                 'mlr_scatter_train.png', 'mlr_scatter_test.png', 'mlr_scatter_combined.png',
                 r2_train, r2_test, rmse_train, rmse_test)

    plot_learning_curve(mlr_model, X_train_poly, y_train, cv_strategy, 'mlr_learning_curve.png', 
                    X_val=X_val_poly, y_val=y_val, exact_val_r2=r2_val)

    print("Printing model equation...")
    print_model_equation(mlr_model, poly_features, selected_features_important_simple, threshold=0.1)

    print("Generating real extrapolation test...")
    X_test_actual, X_test_extrap, y_test_actual, y_test_extrap = train_test_split(X_test, y_test, test_size=0.5, random_state=42)
    X_test_actual_poly, _ = add_polynomial_terms(X_test_actual, degree=best_params['degree'])
    y_pred_test_actual = mlr_model.predict(X_test_actual_poly)
    plot_real_extrapolation_hexbin(y_test_actual, y_pred_test_actual, 'real_extrapolation_plots')

    print("Generating Williams plot...")
    plot_williams(mlr_model, X_train_poly, y_train, X_test_poly, y_test, 'williams_plot.png')

    print("Model training and evaluation completed.")

    save_model = input("Do you want to save the current best model? (y/n): ").lower().strip() == 'y'
    if save_model:
        save_model_to_file(mlr_model, best_params, poly_features, selected_features_important_simple, imputer, scaler, label_transformer, model_path)
        print(f"Model saved to {model_path}")

    end_time = time.time()
    print(f"Program execution completed, total time: {(end_time - start_time) / 60:.2f} minutes")

if __name__ == "__main__":
    main()

Starting main program...
Using 18 CPU threads for parallel computing
Step 1/10: Loading data
Original dataset size: 191
Filtered dataset size: 191
Removed 0 outliers (0.00%)
Filtered data saved to filtered_data.xlsx
Loaded data shape: X: (191, 1), y: (191,)
Step 2/10: Generating features
Number of generated features: 67
Step 3/10: Data preprocessing
Preprocessing completed
Step 4/10: Calculating mutual information and correlations


n_quantiles (1000) is greater than the total number of samples (191). n_quantiles is set to n_samples.


Step 5/10: Plotting feature analysis
Feature analysis plots completed
Step 6/10: Feature selection
Do you want to manually specify features for training? (y/n): n
Selected features:
EState_VSA8
Chi1v
heavy_atom_count
PEOE_VSA14
heteroatom_count
Step 7/10: Preparing final dataset
Step 8/10: Checking previously saved model
Found a previously saved model. Do you want to load it? (y/n): n
Step 9/10: Performing nested cross-validation


Nested cross-validation progress:   1%|▌                                               | 5/400 [00:14<20:11,  3.07s/it][I 2026-05-04 12:26:49,174] Trial 4 finished with value: -2.7399979876167775 and parameters: {'alpha': 9.919110942575018e-05, 'l1_ratio': 0.38251345026091277}. Best is trial 0 with value: 0.5687711279277444.
[I 2026-05-04 12:26:49,186] Trial 5 finished with value: 0.10754530331175902 and parameters: {'alpha': 0.28142315087687686, 'l1_ratio': 0.48257997726726487}. Best is trial 0 with value: 0.5687711279277444.
Nested cross-validation progress:   2%|▉                                               | 8/400 [00:26<26:23,  4.04s/it][I 2026-05-04 12:27:01,623] Trial 7 finished with value: -25.841223783392138 and parameters: {'alpha': 1.7129432061277985e-05, 'l1_ratio': 0.3587790082525607}. Best is trial 0 with value: 0.5687711279277444.
[I 2026-05-04 12:27:01,660] Trial 8 finished with value: 0.10901444874343975 and parameters: {'alpha': 0.06085853529803383, 'l1_ratio': 0.36

Nested cross-validation progress:  10%|████▋                                          | 40/400 [02:28<22:26,  3.74s/it][I 2026-05-04 12:29:03,144] Trial 39 finished with value: -0.46008373936666713 and parameters: {'alpha': 0.00023617020709919238, 'l1_ratio': 0.45818435336058966}. Best is trial 26 with value: 0.6163849686912908.
[I 2026-05-04 12:29:03,160] Trial 40 finished with value: 0.10793920020145398 and parameters: {'alpha': 0.09769299652168235, 'l1_ratio': 0.5654571662834411}. Best is trial 26 with value: 0.6163849686912908.
Nested cross-validation progress:  12%|█████▉                                         | 50/400 [03:01<20:46,  3.56s/it][I 2026-05-04 12:29:36,672] Trial 49 finished with value: 0.5705894488837845 and parameters: {'alpha': 0.009410441055588758, 'l1_ratio': 0.2179413841069022}. Best is trial 26 with value: 0.6163849686912908.


...

Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.6971870247990409, tolerance: 0.10863795777660094
[I 2026-05-04 12:51:14,209] Trial 229 finished with value: 0.6496111262505763 and parameters: {'alpha': 0.0010096239501553549, 'l1_ratio': 0.5146644708531672, 'degree': 4}. Best is trial 0 with value: 0.7740190298918674.
Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.7128199141543352, tolerance: 0.10863795777660094
[I 2026-05-04 12:51:16,102] Trial 230 finished with value: 0.6499889467502475 and parameters: {'alpha': 0.0010330602502282395, 'l1_ratio': 0.4852968459534722, 'degree': 4}. Best is trial 0 with value: 0.7740190298918674.
Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.7112400711477112, tolerance: 0.10863795777660094
[I 2026-05-04 12:51:30,021] Trial 228 finished with value: 0.6495501536155954 and parameters: {'alpha': 0.00103183044512327

Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.7739020764070481, tolerance: 0.10863795777660094
[I 2026-05-04 12:55:16,915] Trial 248 finished with value: 0.6219339364611232 and parameters: {'alpha': 0.0013597675907897249, 'l1_ratio': 0.46347079408458425, 'degree': 4}. Best is trial 0 with value: 0.7740190298918674.
Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.6880649496417726, tolerance: 0.10863795777660094
[I 2026-05-04 12:55:20,930] Trial 249 finished with value: 0.6179658799817369 and parameters: {'alpha': 0.0013806828378934107, 'l1_ratio': 0.49540987806565145, 'degree': 4}. Best is trial 0 with value: 0.7740190298918674.
Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.6635525035142693, tolerance: 0.10863795777660094
[I 2026-05-04 12:55:25,124] Trial 251 finished with value: 0.6194191281396652 and parameters: {'alpha': 0.001330408088676

[I 2026-05-04 12:59:02,264] Trial 277 finished with value: 0.46671365748911986 and parameters: {'alpha': 0.01846620693802517, 'l1_ratio': 0.1524108350320743, 'degree': 4}. Best is trial 0 with value: 0.7740190298918674.
Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.9347781377454565, tolerance: 0.10863795777660094
[I 2026-05-04 12:59:50,929] Trial 254 finished with value: 0.6812960752578046 and parameters: {'alpha': 0.001354554771006642, 'l1_ratio': 0.1548570981652084, 'degree': 4}. Best is trial 0 with value: 0.7740190298918674.
Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.9697209315468918, tolerance: 0.10863795777660094
[I 2026-05-04 12:59:54,128] Trial 271 finished with value: 0.6526097923100269 and parameters: {'alpha': 0.0016964285998377388, 'l1_ratio': 0.15061736219641697, 'degree': 4}. Best is trial 0 with value: 0.7740190298918674.
Objective did not converge. You might want to in

Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.979767593468568, tolerance: 0.10863795777660094
[I 2026-05-04 13:03:19,356] Trial 280 finished with value: 0.6978777810136558 and parameters: {'alpha': 0.0011865892923979295, 'l1_ratio': 0.15206362741762441, 'degree': 4}. Best is trial 0 with value: 0.7740190298918674.
Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.6546962556809679, tolerance: 0.10863795777660094
[I 2026-05-04 13:03:21,585] Trial 282 finished with value: 0.5700257465564813 and parameters: {'alpha': 0.003693231699499151, 'l1_ratio': 0.14994982523158873, 'degree': 4}. Best is trial 0 with value: 0.7740190298918674.
Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.9241906275906515, tolerance: 0.10863795777660094
[I 2026-05-04 13:03:35,641] Trial 288 finished with value: 0.6933009265981274 and parameters: {'alpha': 0.00119496042425306

New trained model's test set R² score: 0.773
Do you want to save the new model? (y/n): n
Final MLR parameters:
alpha: 0.000262538285619946
l1_ratio: 0.5293154887169182
degree: 3
Performing final evaluation...
Training R²: 0.810
Validation R²: 0.774
Testing R²: 0.773
Training RMSE: 0.123
Validation RMSE: 0.143
Testing RMSE: 0.145
Performing cross-validation...
Plotting results...

Learning Curve Generated Successfully:
Training R²: 0.6000 → 0.8102
Validation R²: 0.4000 → 0.7740
Printing model equation...
Debug: Original feature names: ['EState_VSA8', 'Chi1v', 'heavy_atom_count', 'PEOE_VSA14', 'heteroatom_count']
Debug: Feature map: {'x0': 'EState_VSA8', 'x1': 'Chi1v', 'x2': 'heavy_atom_count', 'x3': 'PEOE_VSA14', 'x4': 'heteroatom_count'}
Debug: Original feature: x0, Mapped name: EState_VSA8
Debug: Original feature: x1, Mapped name: Chi1v
Debug: Original feature: x3, Mapped name: PEOE_VSA14
Debug: Original feature: x4, Mapped name: heteroatom_count
Debug: Original feature: x0^2, Mapped 